In [2]:
import pandas as pd
from pathlib import Path

DATA = Path(r"C:\Users\Dell\Desktop\carnet_data")
pd.set_option("display.max_columns", 50, "display.width", 200)

def inspect(df, name, label=None):
    print(f"=== {name} ===")
    print("shape:", df.shape)
    print("duplicate rows:", df.duplicated().sum())
    print("\ncolumns / dtype / missing:")
    print(pd.DataFrame({"dtype": df.dtypes, "missing": df.isna().sum(),
                        "missing_%": (df.isna().mean() * 100).round(1)}))
    if label:
        print(f"\nlabel counts ({label}):")
        print(df[label].value_counts(dropna=False))
    return df.head(3)

Resume

In [3]:
resume = pd.read_csv(DATA / "resume" / "Resume dataset.csv")
inspect(resume, "resume", label="category")
print("duplicate texts:", resume["Text"].duplicated().sum())
print(resume["Text"].str.split().str.len().describe())

=== resume ===
shape: (9000, 3)
duplicate rows: 2156

columns / dtype / missing:
          dtype  missing  missing_%
category    str        0        0.0
job_title   str        0        0.0
Text        str        0        0.0

label counts (category):
category
Java Developers/Architects Resumes                1000
Web Developer Resumes                             1000
SQL Developers Resumes                            1000
Business Analyst (BA) Resumes                     1000
Network and Systems Administrators Resumes        1000
Datawarehousing, ETL, Informatica Resumes         1000
Business Intelligence, Business Object Resumes    1000
Project Manager Resumes                           1000
Recruiter Resumes                                 1000
Name: count, dtype: int64
duplicate texts: 3259
count    9000.000000
mean     1618.590778
std       840.723871
min        76.000000
25%      1041.500000
50%      1542.000000
75%      2060.000000
max      8341.000000
Name: Text, dtype: float64


In [4]:
import pandas as pd
import numpy as np
import re
import html
import unicodedata

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# 1. Copy dataset
df = resume.copy()

# 2. Remove exact duplicate rows
df = df.drop_duplicates().reset_index(drop=True)

# 3. Remove missing values
df = df.dropna(subset=["category", "Text"])

# 4. Clean resume text
def clean_text(text):
    text = str(text)
    text = unicodedata.normalize("NFKC", text)
    text = html.unescape(text)
    text = text.lower()

    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"\+?\d[\d\s().-]{7,}\d", " ", text)

    # Preserve technical terms such as C++, C#, .NET
    text = re.sub(r"[^a-z0-9+#./\s-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

df["clean_text"] = df["Text"].apply(clean_text)

# 5. Remove empty texts
df = df[df["clean_text"].str.len() > 0].copy()

# 6. Identify conflicting labels
label_counts = df.groupby("clean_text")["category"].nunique()
conflicting_texts = label_counts[label_counts > 1].index

print("Conflicting texts:", len(conflicting_texts))

# Remove texts assigned to multiple categories
df = df[~df["clean_text"].isin(conflicting_texts)].copy()

# 7. Remove duplicate resume texts
before = len(df)
df = df.drop_duplicates(subset=["clean_text"]).reset_index(drop=True)

print("Duplicate texts removed:", before - len(df))

# 8. Encode categories
label_encoder = LabelEncoder()

X = df["clean_text"]
y = label_encoder.fit_transform(df["category"])

# 9. Split dataset: 70% train, 15% validation, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

# 10. Display results
print("\n========== PREPROCESSING RESULTS ==========")
print("Original dataset:", resume.shape)
print("Final dataset:", df.shape)
print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Testing samples:", len(X_test))

print("\nCategory mapping:")
for i, category in enumerate(label_encoder.classes_):
    print(i, ":", category)

print("\nFinal class distribution:")
print(df["category"].value_counts())

print("\nMissing values:")
print(df[["clean_text", "category"]].isnull().sum())

print("\nRemaining duplicate texts:", df["clean_text"].duplicated().sum())

Conflicting texts: 310
Duplicate texts removed: 787

========== PREPROCESSING RESULTS ==========
Original dataset: (9000, 3)
Final dataset: (5295, 4)
Training samples: 3706
Validation samples: 794
Testing samples: 795

Category mapping:
0 : Business Analyst (BA) Resumes
1 : Business Intelligence, Business Object Resumes
2 : Datawarehousing, ETL, Informatica Resumes
3 : Java Developers/Architects Resumes
4 : Network and Systems Administrators Resumes
5 : Project Manager Resumes
6 : Recruiter Resumes
7 : SQL Developers Resumes
8 : Web Developer Resumes

Final class distribution:
category
Datawarehousing, ETL, Informatica Resumes         878
SQL Developers Resumes                            729
Business Intelligence, Business Object Resumes    692
Web Developer Resumes                             671
Project Manager Resumes                           638
Network and Systems Administrators Resumes        636
Business Analyst (BA) Resumes                     556
Recruiter Resumes            

In [5]:
class_distribution = df["category"].value_counts().to_frame("Samples")

class_distribution["Percentage"] = (
    class_distribution["Samples"] / len(df) * 100
).round(2)

display(class_distribution)

print("Largest class:", class_distribution["Samples"].max())
print("Smallest class:", class_distribution["Samples"].min())

,Samples,Percentage
category,,
"Datawarehousing, ETL, Informatica Resumes",878,16.58
SQL Developers Resumes,729,13.77
"Business Intelligence, Business Object Resumes",692,13.07
Web Developer Resumes,671,12.67
Project Manager Resumes,638,12.05
Network and Systems Administrators Resumes,636,12.01
Business Analyst (BA) Resumes,556,10.50
Recruiter Resumes,312,5.89
Java Developers/Architects Resumes,183,3.46


Largest class: 878
Smallest class: 183


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

# Fit only on training data
X_train_tfidf = tfidf.fit_transform(X_train)

# Transform validation and test data
X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Validation TF-IDF shape:", X_val_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)
print("Vocabulary size:", len(tfidf.vocabulary_))

Training TF-IDF shape: (3706, 30000)
Validation TF-IDF shape: (794, 30000)
Testing TF-IDF shape: (795, 30000)
Vocabulary size: 30000


In [7]:

import os
import joblib
from scipy.sparse import save_npz

# 1. Define paths
RAW_DATA = r"C:\Users\Dell\Desktop\carnet_data"
FYP_DIR = r"C:\Users\Dell\Desktop\founders-carnet"

PREPROCESSED_DIR = os.path.join(
    FYP_DIR, "classifier_data", "preprocessed"
)

# 2. Create classifier folders
for folder in [
    "resume_classifier",
    "startup_success_classifier",
    "startup_investment_classifier"
]:
    os.makedirs(
        os.path.join(PREPROCESSED_DIR, folder),
        exist_ok=True
    )

RESUME_DIR = os.path.join(
    PREPROCESSED_DIR, "resume_classifier"
)

# 3. Save cleaned dataset
df.to_csv(
    os.path.join(RESUME_DIR, "cleaned_resume.csv"),
    index=False
)

# 4. Save data splits
joblib.dump({
    "X_train": X_train,
    "X_val": X_val,
    "X_test": X_test,
    "y_train": y_train,
    "y_val": y_val,
    "y_test": y_test
}, os.path.join(RESUME_DIR, "data_splits.pkl"))

# 5. Save TF-IDF matrices
save_npz(os.path.join(RESUME_DIR, "X_train_tfidf.npz"), X_train_tfidf)
save_npz(os.path.join(RESUME_DIR, "X_val_tfidf.npz"), X_val_tfidf)
save_npz(os.path.join(RESUME_DIR, "X_test_tfidf.npz"), X_test_tfidf)

# 6. Save vectorizer and label encoder
joblib.dump(tfidf, os.path.join(RESUME_DIR, "tfidf_vectorizer.pkl"))
joblib.dump(label_encoder, os.path.join(RESUME_DIR, "label_encoder.pkl"))

# 7. Confirm saved files
print("Resume preprocessing data saved successfully!")
print("Location:", RESUME_DIR)

print("\nSaved files:")
for file in sorted(os.listdir(RESUME_DIR)):
    print("-", file)


Resume preprocessing data saved successfully!
Location: C:\Users\Dell\Desktop\founders-carnet\classifier_data\preprocessed\resume_classifier

Saved files:
- X_test_tfidf.npz
- X_train_tfidf.npz
- X_val_tfidf.npz
- cleaned_resume.csv
- data_splits.pkl
- label_encoder.pkl
- tfidf_vectorizer.pkl


Startup

In [8]:
outcome = pd.read_csv(DATA / "outcome" / "big_startup_secsees_dataset.csv")
inspect(outcome, "startup outcome", label="status")   # change label if the column differs

=== startup outcome ===
shape: (66368, 14)
duplicate rows: 0

columns / dtype / missing:
                   dtype  missing  missing_%
permalink            str        0        0.0
name                 str        1        0.0
homepage_url         str     5058        7.6
category_list        str     3148        4.7
funding_total_usd    str        0        0.0
status               str        0        0.0
country_code         str     6958       10.5
state_code           str     8547       12.9
region               str     8030       12.1
city                 str     8028       12.1
funding_rounds     int64        0        0.0
founded_at           str    15221       22.9
first_funding_at     str       24        0.0
last_funding_at      str        0        0.0

label counts (status):
status
operating    53034
closed        6238
acquired      5549
ipo           1547
Name: count, dtype: int64


,permalink,name,homepage_url,category_list,funding_total_usd,status,country_code,state_code,region,city,funding_rounds,founded_at,first_funding_at,last_funding_at
0,/organization/-fame,#fame,http://livfame.com,Media,10000000,operating,IND,16,Mumbai,Mumbai,1,NaN,2015-01-05,2015-01-05
1,/organization/-qounter,:Qounter,http://www.qounter.com,Application Platforms|Real Time|Social Network...,700000,operating,USA,DE,DE - Other,Delaware City,2,2014-09-04,2014-03-01,2014-10-14
2,/organization/-the-one-of-them-inc-,"(THE) ONE of THEM,Inc.",http://oneofthem.jp,Apps|Games|Mobile,3406878,operating,NaN,NaN,NaN,NaN,1,NaN,2014-01-30,2014-01-30


In [21]:

# ============================================
# STARTUP OUTCOME CLASSIFIER
# PREPROCESSING
# ============================================

import os
import glob
import numpy as np
import pandas as pd
import scipy.sparse as sp
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

# --------------------------------------------
# 1. SETTINGS
# --------------------------------------------

RANDOM_STATE = 42

DATA_DIR = r"C:\Users\Dell\Desktop\carnet_data\outcome"

OUTPUT_DIR = (
    r"C:\Users\Dell\Desktop\founders-carnet"
    r"\classifier_data\preprocessed"
    r"\startup_outcome_classifier"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Automatically locate the dataset
files = glob.glob(
    os.path.join(DATA_DIR, "big_startup_secsees_dataset.*")
)

if len(files) == 0:
    raise FileNotFoundError(
        "Dataset file not found in: " + DATA_DIR
    )

if len(files) > 1:
    raise ValueError(
        f"Multiple matching files found: {files}"
    )

DATA_PATH = files[0]

# Load according to file extension
if DATA_PATH.lower().endswith(".csv"):
    startup = pd.read_csv(DATA_PATH)
elif DATA_PATH.lower().endswith((".xlsx", ".xls")):
    startup = pd.read_excel(DATA_PATH)
else:
    raise ValueError("Unsupported file format: " + DATA_PATH)

print("Loaded:", DATA_PATH)
print("Original shape:", startup.shape)

# --------------------------------------------
# 2. CHECK REQUIRED COLUMNS
# --------------------------------------------

required_columns = [
    "category_list",
    "country_code",
    "state_code",
    "region",
    "city",
    "founded_at",
    "first_funding_at",
    "status"
]

missing_columns = [
    col for col in required_columns
    if col not in startup.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

# --------------------------------------------
# 3. CLEAN STATUS AND CREATE TARGET
# --------------------------------------------

df = startup.copy().reset_index(drop=True)

df["status"] = (
    df["status"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

# 1 = Successful
# 0 = Failed

status_mapping = {
    "operating": 1,
    "acquired": 1,
    "ipo": 1,
    "closed": 0
}

df["target"] = df["status"].map(status_mapping)

# Remove missing or unrecognized statuses
df = df.dropna(subset=["target"]).copy()

df["target"] = df["target"].astype(int)

# --------------------------------------------
# 4. CLEAN INVALID DATES
# --------------------------------------------

df["founded_at"] = pd.to_datetime(
    df["founded_at"],
    errors="coerce"
)

df["first_funding_at"] = pd.to_datetime(
    df["first_funding_at"],
    errors="coerce"
)

# Extract founded year
df["founded_year"] = df["founded_at"].dt.year

# Extract first funding year and month
df["first_funding_year"] = (
    df["first_funding_at"].dt.year
)

df["first_funding_month"] = (
    df["first_funding_at"].dt.month
)

# Remove implausible founding years
df.loc[
    (df["founded_year"] < 1800) |
    (df["founded_year"] > 2015),
    "founded_year"
] = np.nan

# --------------------------------------------
# 5. SELECT FEATURES
# --------------------------------------------

text_feature = "category_list"

categorical_features = [
    "country_code",
    "state_code",
    "region",
    "city"
]

numeric_features = [
    "founded_year",
    "first_funding_year",
    "first_funding_month"
]

feature_columns = (
    [text_feature] +
    categorical_features +
    numeric_features
)

X = df[feature_columns].copy()
y = df["target"].copy()

# Clean text and categorical values
for col in [text_feature] + categorical_features:
    X[col] = (
        X[col]
        .fillna("")
        .astype(str)
        .str.strip()
    )

# --------------------------------------------
# 6. SPLIT DATA
# --------------------------------------------

# 70% training, 15% validation, 15% testing

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

# --------------------------------------------
# 7. BUILD PREPROCESSING PIPELINE
# --------------------------------------------

text_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=10000,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
        token_pattern=r"(?u)\b\w+\b"
    ))
])

categorical_pipeline = Pipeline([
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=10
    ))
])

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="median"
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("category_text", text_pipeline, text_feature),
        ("categorical", categorical_pipeline,
         categorical_features),
        ("numeric", numeric_pipeline, numeric_features)
    ],
    remainder="drop",
    sparse_threshold=1.0
)

# --------------------------------------------
# 8. FIT ON TRAINING DATA ONLY
# --------------------------------------------

X_train_processed = preprocessor.fit_transform(
    X_train
)

X_val_processed = preprocessor.transform(
    X_val
)

X_test_processed = preprocessor.transform(
    X_test
)

# Convert to CSR sparse matrices
X_train_processed = sp.csr_matrix(X_train_processed)
X_val_processed = sp.csr_matrix(X_val_processed)
X_test_processed = sp.csr_matrix(X_test_processed)

# --------------------------------------------
# 9. SAVE PROCESSED MATRICES
# --------------------------------------------

sp.save_npz(
    os.path.join(OUTPUT_DIR, "X_train.npz"),
    X_train_processed
)

sp.save_npz(
    os.path.join(OUTPUT_DIR, "X_val.npz"),
    X_val_processed
)

sp.save_npz(
    os.path.join(OUTPUT_DIR, "X_test.npz"),
    X_test_processed
)

# --------------------------------------------
# 10. SAVE TARGET LABELS
# --------------------------------------------

y_train.to_csv(
    os.path.join(OUTPUT_DIR, "y_train.csv"),
    index=False
)

y_val.to_csv(
    os.path.join(OUTPUT_DIR, "y_val.csv"),
    index=False
)

y_test.to_csv(
    os.path.join(OUTPUT_DIR, "y_test.csv"),
    index=False
)

# --------------------------------------------
# 11. SAVE PREPROCESSING OBJECTS
# --------------------------------------------

joblib.dump(
    preprocessor,
    os.path.join(OUTPUT_DIR, "preprocessor.pkl")
)

joblib.dump(
    {
        "feature_columns": feature_columns,
        "text_feature": text_feature,
        "categorical_features": categorical_features,
        "numeric_features": numeric_features,
        "target_name": "target",
        "target_mapping": {
            0: "Failed (Closed)",
            1: "Successful (Operating, Acquired, IPO)"
        },
        "random_state": RANDOM_STATE
    },
    os.path.join(OUTPUT_DIR, "metadata.pkl")
)

# Save cleaned dataset
df.to_csv(
    os.path.join(OUTPUT_DIR, "cleaned_startup_outcome.csv"),
    index=False
)

# --------------------------------------------
# 12. DISPLAY RESULTS
# --------------------------------------------

print("\n" + "=" * 55)
print("STARTUP OUTCOME PREPROCESSING COMPLETE")
print("=" * 55)

print("\nOriginal dataset shape:", startup.shape)
print("Final dataset shape:", df.shape)

print("\nOriginal status distribution:")
print(startup["status"].value_counts(dropna=False))

print("\nTarget distribution:")
print(
    df["target"].value_counts()
    .sort_index()
    .rename(index={
        0: "Failed (Closed)",
        1: "Successful"
    })
)

print("\nTarget percentages:")
print(
    (df["target"].value_counts(normalize=True) * 100)
    .sort_index()
    .round(2)
)

print("\nSplit sizes:")
print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Testing:", len(X_test))

print("\nProcessed matrix shapes:")
print("Training:", X_train_processed.shape)
print("Validation:", X_val_processed.shape)
print("Testing:", X_test_processed.shape)

print("\nMissing date values after cleaning:")
print("Founded year:", df["founded_year"].isna().sum())
print("First funding year:", df["first_funding_year"].isna().sum())
print("First funding month:", df["first_funding_month"].isna().sum())

print("\nSaved files:")
for file in sorted(os.listdir(OUTPUT_DIR)):
    print("-", file)

print("\nOutput directory:")
print(OUTPUT_DIR)

print("=" * 55)


Loaded: C:\Users\Dell\Desktop\carnet_data\outcome\big_startup_secsees_dataset.csv
Original shape: (66368, 14)

STARTUP OUTCOME PREPROCESSING COMPLETE

Original dataset shape: (66368, 14)
Final dataset shape: (66368, 18)

Original status distribution:
status
operating    53034
closed        6238
acquired      5549
ipo           1547
Name: count, dtype: int64

Target distribution:
target
Failed (Closed)     6238
Successful         60130
Name: count, dtype: int64

Target percentages:
target
0     9.4
1    90.6
Name: proportion, dtype: float64

Split sizes:
Training: 46457
Validation: 9955
Testing: 9956

Processed matrix shapes:
Training: (46457, 9292)
Validation: (9955, 9292)
Testing: (9956, 9292)

Missing date values after cleaning:
Founded year: 15235
First funding year: 24
First funding month: 24

Saved files:
- X_test.npz
- X_train.npz
- X_val.npz
- cleaned_startup_outcome.csv
- metadata.pkl
- preprocessor.pkl
- y_test.csv
- y_train.csv
- y_val.csv

Output directory:
C:\Users\Dell\Des

In [22]:
import os
import scipy.sparse as sp

OUTPUT_DIR = (
    r"C:\Users\Dell\Desktop\founders-carnet"
    r"\classifier_data\preprocessed"
    r"\startup_outcome_classifier"
)

for name in ["X_train.npz", "X_val.npz", "X_test.npz"]:
    matrix = sp.load_npz(os.path.join(OUTPUT_DIR, name))
    print(name, matrix.shape)

print("\nSplit sizes:")
for name in ["y_train.csv", "y_val.csv", "y_test.csv"]:
    labels = pd.read_csv(os.path.join(OUTPUT_DIR, name))
    print(name, len(labels), labels.iloc[:, 0].value_counts().to_dict())

print("\nSaved files:")
for file in sorted(os.listdir(OUTPUT_DIR)):
    print(file)

X_train.npz (46457, 9292)
X_val.npz (9955, 9292)
X_test.npz (9956, 9292)

Split sizes:
y_train.csv 46457 {1: 42090, 0: 4367}
y_val.csv 9955 {1: 9020, 0: 935}
y_test.csv 9956 {1: 9020, 0: 936}

Saved files:
X_test.npz
X_train.npz
X_val.npz
cleaned_startup_outcome.csv
metadata.pkl
preprocessor.pkl
y_test.csv
y_train.csv
y_val.csv


Funding

In [9]:
funding = pd.read_csv(DATA / "funding" / "investments_VC.csv", encoding="latin-1")
funding.columns = funding.columns.str.strip()
inspect(funding, "funding")
print(funding[["seed", "round_A", "funding_rounds", "status"]].describe(include="all"))

=== funding ===
shape: (54294, 39)
duplicate rows: 4855

columns / dtype / missing:
                        dtype  missing  missing_%
permalink                 str     4856        8.9
name                      str     4857        8.9
homepage_url              str     8305       15.3
category_list             str     8817       16.2
market                    str     8824       16.3
funding_total_usd         str     4856        8.9
status                    str     6170       11.4
country_code              str    10129       18.7
state_code                str    24133       44.4
region                    str    10129       18.7
city                      str    10972       20.2
funding_rounds        float64     4856        8.9
founded_at                str    15740       29.0
founded_month             str    15812       29.1
founded_quarter           str    15812       29.1
founded_year          float64    15812       29.1
first_funding_at          str     4856        8.9
last_funding_at 

In [10]:
# Inspect all funding dataset columns
print("Dataset shape:", funding.shape)

print("\nAll columns:")
for i, col in enumerate(funding.columns, 1):
    print(i, col)

print("\nData types:")
print(funding.dtypes)

print("\nFirst 5 records:")
display(funding.head())

print("\nLast 10 columns (funding-related fields):")
display(funding.iloc[:, -10:].head())

print("\nMissing values:")
display(
    funding.isnull().sum()
    .sort_values(ascending=False)
    .to_frame("Missing Values")
)

Dataset shape: (54294, 39)

All columns:
1 permalink
2 name
3 homepage_url
4 category_list
5 market
6 funding_total_usd
7 status
8 country_code
9 state_code
10 region
11 city
12 funding_rounds
13 founded_at
14 founded_month
15 founded_quarter
16 founded_year
17 first_funding_at
18 last_funding_at
19 seed
20 venture
21 equity_crowdfunding
22 undisclosed
23 convertible_note
24 debt_financing
25 angel
26 grant
27 private_equity
28 post_ipo_equity
29 post_ipo_debt
30 secondary_market
31 product_crowdfunding
32 round_A
33 round_B
34 round_C
35 round_D
36 round_E
37 round_F
38 round_G
39 round_H

Data types:
permalink                   str
name                        str
homepage_url                str
category_list               str
market                      str
funding_total_usd           str
status                      str
country_code                str
state_code                  str
region                      str
city                        str
funding_rounds          float64
founde

,permalink,name,homepage_url,category_list,market,funding_total_usd,status,country_code,state_code,region,city,funding_rounds,founded_at,founded_month,founded_quarter,founded_year,first_funding_at,last_funding_at,seed,venture,equity_crowdfunding,undisclosed,convertible_note,debt_financing,angel,grant,private_equity,post_ipo_equity,post_ipo_debt,secondary_market,product_crowdfunding,round_A,round_B,round_C,round_D,round_E,round_F,round_G,round_H
0,/organization/waywire,#waywire,http://www.waywire.com,|Entertainment|Politics|Social Media|News|,News,"17,50,000",acquired,USA,NY,New York City,New York,1.0,2012-06-01,2012-06,2012-Q2,2012.0,2012-06-30,2012-06-30,1750000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,/organization/tv-communications,&TV Communications,http://enjoyandtv.com,|Games|,Games,"40,00,000",operating,USA,CA,Los Angeles,Los Angeles,2.0,NaN,NaN,NaN,NaN,2010-06-04,2010-09-23,0.0,4000000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,/organization/rock-your-paper,'Rock' Your Paper,http://www.rockyourpaper.org,|Publishing|Education|,Publishing,"40,000",operating,EST,NaN,Tallinn,Tallinn,1.0,2012-10-26,2012-10,2012-Q4,2012.0,2012-08-09,2012-08-09,40000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,/organization/in-touch-network,(In)Touch Network,http://www.InTouchNetwork.com,|Electronics|Guides|Coffee|Restaurants|Music|i...,Electronics,"15,00,000",operating,GBR,NaN,London,London,1.0,2011-04-01,2011-04,2011-Q2,2011.0,2011-04-01,2011-04-01,1500000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,/organization/r-ranch-and-mine,-R- Ranch and Mine,NaN,|Tourism|Entertainment|Games|,Tourism,"60,000",operating,USA,TX,Dallas,Fort Worth,2.0,2014-01-01,2014-01,2014-Q1,2014.0,2014-08-17,2014-09-26,0.0,0.0,60000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



Last 10 columns (funding-related fields):


,secondary_market,product_crowdfunding,round_A,round_B,round_C,round_D,round_E,round_F,round_G,round_H
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



Missing values:


,Missing Values
state_code,24133
founded_quarter,15812
founded_month,15812
founded_year,15812
founded_at,15740
city,10972
region,10129
country_code,10129
market,8824
category_list,8817


In [11]:
# 1. Identify funding-related columns
metadata_cols = [
    "permalink", "name", "homepage_url", "category_list",
    "market", "funding_total_usd", "status", "country_code",
    "state_code", "region", "city", "funding_rounds",
    "founded_at", "founded_month", "founded_quarter",
    "founded_year", "first_funding_at", "last_funding_at"
]

funding_cols = [col for col in funding.columns if col not in metadata_cols]

print("FUNDING COLUMNS:")
print(funding_cols)

# 2. Check rows with all funding information missing
print("\nRows with all funding columns missing:",
      funding[funding_cols].isnull().all(axis=1).sum())

# 3. Check rows with no recorded total funding
print("Rows with missing total funding:",
      funding["funding_total_usd"].isnull().sum())

# 4. Funding column statistics
print("\nFUNDING COLUMN STATISTICS:")
display(funding[funding_cols].describe().T)

# 5. Check status distribution
print("\nSTATUS DISTRIBUTION:")
display(funding["status"].value_counts(dropna=False))

# 6. Check sample rows with missing funding information
display(funding[funding["funding_total_usd"].isnull()].head(10))

FUNDING COLUMNS:
['seed', 'venture', 'equity_crowdfunding', 'undisclosed', 'convertible_note', 'debt_financing', 'angel', 'grant', 'private_equity', 'post_ipo_equity', 'post_ipo_debt', 'secondary_market', 'product_crowdfunding', 'round_A', 'round_B', 'round_C', 'round_D', 'round_E', 'round_F', 'round_G', 'round_H']

Rows with all funding columns missing: 4856
Rows with missing total funding: 4856

FUNDING COLUMN STATISTICS:


,count,mean,std,min,25%,50%,75%,max
seed,49438.0,2.173215e+05,1.056985e+06,0.0,0.0,0.0,25000.0,1.300000e+08
venture,49438.0,7.501051e+06,2.847112e+07,0.0,0.0,0.0,5000000.0,2.351000e+09
equity_crowdfunding,49438.0,6.163322e+03,1.999048e+05,0.0,0.0,0.0,0.0,2.500000e+07
undisclosed,49438.0,1.302213e+05,2.981404e+06,0.0,0.0,0.0,0.0,2.924328e+08
convertible_note,49438.0,2.336410e+04,1.432046e+06,0.0,0.0,0.0,0.0,3.000000e+08
debt_financing,49438.0,1.888157e+06,1.382046e+08,0.0,0.0,0.0,0.0,3.007950e+10
angel,49438.0,6.541898e+04,6.582908e+05,0.0,0.0,0.0,0.0,6.359026e+07
grant,49438.0,1.628453e+05,5.612088e+06,0.0,0.0,0.0,0.0,7.505000e+08
private_equity,49438.0,2.074286e+06,3.167231e+07,0.0,0.0,0.0,0.0,3.500000e+09
post_ipo_equity,49438.0,6.088736e+05,2.678348e+07,0.0,0.0,0.0,0.0,4.700000e+09



STATUS DISTRIBUTION:


status
operating    41829
NaN           6170
acquired      3692
closed        2603
Name: count, dtype: int64

,permalink,name,homepage_url,category_list,market,funding_total_usd,status,country_code,state_code,region,city,funding_rounds,founded_at,founded_month,founded_quarter,founded_year,first_funding_at,last_funding_at,seed,venture,equity_crowdfunding,undisclosed,convertible_note,debt_financing,angel,grant,private_equity,post_ipo_equity,post_ipo_debt,secondary_market,product_crowdfunding,round_A,round_B,round_C,round_D,round_E,round_F,round_G,round_H
49438,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49439,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49440,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49441,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49442,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49443,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49444,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49445,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49446,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49447,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
# Remove rows where every column is missing
funding = funding.dropna(how="all").reset_index(drop=True)

print("Shape after removing empty rows:", funding.shape)

print("\nDuplicate rows:", funding.duplicated().sum())

print("\nMissing values:")
display(
    funding.isnull().sum()
    .sort_values(ascending=False)
    .to_frame("Missing Values")
)

print("\nStatus distribution:")
display(funding["status"].value_counts(dropna=False))

Shape after removing empty rows: (49438, 39)

Duplicate rows: 0

Missing values:


,Missing Values
state_code,19277
founded_quarter,10956
founded_month,10956
founded_year,10956
founded_at,10884
city,6116
region,5273
country_code,5273
market,3968
category_list,3961



Status distribution:


status
operating    41829
acquired      3692
closed        2603
NaN           1314
Name: count, dtype: int64

In [13]:
# Funding round columns
round_cols = [
    "round_A", "round_B", "round_C", "round_D",
    "round_E", "round_F", "round_G", "round_H"
]

# Count companies with funding in each round
print("COMPANIES WITH FUNDING IN EACH ROUND:\n")

for col in round_cols:
    count = (funding[col] > 0).sum()
    print(f"{col}: {count}")

# Count companies with multiple funding rounds
funding["rounds_count"] = (funding[round_cols] > 0).sum(axis=1)

print("\nNUMBER OF COMPANIES BY NUMBER OF FUNDED ROUNDS:")
display(funding["rounds_count"].value_counts().sort_index())

# Companies without any Series A-H funding
print(
    "\nCompanies without Series A-H funding:",
    (funding["rounds_count"] == 0).sum()
)

COMPANIES WITH FUNDING IN EACH ROUND:

round_A: 9003
round_B: 5447
round_C: 2837
round_D: 1288
round_E: 516
round_F: 172
round_G: 34
round_H: 4

NUMBER OF COMPANIES BY NUMBER OF FUNDED ROUNDS:


rounds_count
0    36680
1     8518
2     2699
3     1010
4      355
5      128
6       42
7        5
8        1
Name: count, dtype: int64


Companies without Series A-H funding: 36680


In [14]:
# Early funding columns
early_cols = [
    "seed",
    "venture",
    "angel",
    "equity_crowdfunding",
    "convertible_note",
    "undisclosed",
    "debt_financing",
    "grant",
    "private_equity",
    "post_ipo_equity",
    "post_ipo_debt",
    "secondary_market",
    "product_crowdfunding"
]

# Companies without Series A-H funding
no_series = funding[funding["rounds_count"] == 0]

print("Companies without Series A-H funding:", len(no_series))

print("\nCompanies with early funding:")
for col in early_cols:
    print(col, ":", (no_series[col] > 0).sum())

print("\nCompanies with no early funding either:")
print((no_series[early_cols] == 0).all(axis=1).sum())

print("\nStatus distribution for companies without Series A-H:")
display(no_series["status"].value_counts(dropna=False))

Companies without Series A-H funding: 36680

Companies with early funding:
seed : 12085
venture : 10528
angel : 2574
equity_crowdfunding : 512
convertible_note : 441
undisclosed : 814
debt_financing : 2823
grant : 996
private_equity : 1050
post_ipo_equity : 271
post_ipo_debt : 64
secondary_market : 9
product_crowdfunding : 190

Companies with no early funding either:
8531

Status distribution for companies without Series A-H:


status
operating    31954
closed        1922
acquired      1755
NaN           1049
Name: count, dtype: int64

In [16]:
import numpy as np

# Create target based on highest recorded Series A-H round
conditions = [
    funding["round_H"] > 0,
    funding["round_G"] > 0,
    funding["round_F"] > 0,
    funding["round_E"] > 0,
    funding["round_D"] > 0,
    funding["round_C"] > 0,
    funding["round_B"] > 0,
    funding["round_A"] > 0
]

choices = [
    "Series D+",
    "Series D+",
    "Series D+",
    "Series D+",
    "Series D+",
    "Series C",
    "Series B",
    "Series A"
]

funding["target"] = np.select(
    conditions,
    choices,
    default="No Series A-H"
)

# Check target distribution
print("Target distribution:")
display(funding["target"].value_counts())

print("\nTarget percentages:")
display(
    (funding["target"].value_counts(normalize=True) * 100)
    .round(2)
)

Target distribution:


target
No Series A-H    36680
Series A          5722
Series B          3580
Series C          1925
Series D+         1531
Name: count, dtype: int64


Target percentages:


target
No Series A-H    74.19
Series A         11.57
Series B          7.24
Series C          3.89
Series D+         3.10
Name: proportion, dtype: float64

In [17]:
# Inspect all columns and their data types
print("DATASET INFORMATION:")
funding.info()

print("\nSAMPLE OF CURRENT DATA:")
display(funding.head(10))

print("\nTARGET DISTRIBUTION:")
display(funding["target"].value_counts())

print("\nUNIQUE VALUES IN CATEGORICAL COLUMNS:")
for col in funding.select_dtypes(include=["object", "str"]).columns:
    print(f"\n{col}: {funding[col].nunique()} unique values")
    print(funding[col].value_counts().head(10))

DATASET INFORMATION:
<class 'pandas.DataFrame'>
RangeIndex: 49438 entries, 0 to 49437
Data columns (total 41 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   permalink             49438 non-null  str    
 1   name                  49437 non-null  str    
 2   homepage_url          45989 non-null  str    
 3   category_list         45477 non-null  str    
 4   market                45470 non-null  str    
 5   funding_total_usd     49438 non-null  str    
 6   status                48124 non-null  str    
 7   country_code          44165 non-null  str    
 8   state_code            30161 non-null  str    
 9   region                44165 non-null  str    
 10  city                  43322 non-null  str    
 11  funding_rounds        49438 non-null  float64
 12  founded_at            38554 non-null  str    
 13  founded_month         38482 non-null  str    
 14  founded_quarter       38482 non-null  str    
 15  founded_y

,permalink,name,homepage_url,category_list,market,funding_total_usd,status,country_code,state_code,region,city,funding_rounds,founded_at,founded_month,founded_quarter,founded_year,first_funding_at,last_funding_at,seed,venture,equity_crowdfunding,undisclosed,convertible_note,debt_financing,angel,grant,private_equity,post_ipo_equity,post_ipo_debt,secondary_market,product_crowdfunding,round_A,round_B,round_C,round_D,round_E,round_F,round_G,round_H,rounds_count,target
0,/organization/waywire,#waywire,http://www.waywire.com,|Entertainment|Politics|Social Media|News|,News,"17,50,000",acquired,USA,NY,New York City,New York,1.0,2012-06-01,2012-06,2012-Q2,2012.0,2012-06-30,2012-06-30,1750000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,No Series A-H
1,/organization/tv-communications,&TV Communications,http://enjoyandtv.com,|Games|,Games,"40,00,000",operating,USA,CA,Los Angeles,Los Angeles,2.0,NaN,NaN,NaN,NaN,2010-06-04,2010-09-23,0.0,4000000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,No Series A-H
2,/organization/rock-your-paper,'Rock' Your Paper,http://www.rockyourpaper.org,|Publishing|Education|,Publishing,"40,000",operating,EST,NaN,Tallinn,Tallinn,1.0,2012-10-26,2012-10,2012-Q4,2012.0,2012-08-09,2012-08-09,40000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,No Series A-H
3,/organization/in-touch-network,(In)Touch Network,http://www.InTouchNetwork.com,|Electronics|Guides|Coffee|Restaurants|Music|i...,Electronics,"15,00,000",operating,GBR,NaN,London,London,1.0,2011-04-01,2011-04,2011-Q2,2011.0,2011-04-01,2011-04-01,1500000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,No Series A-H
4,/organization/r-ranch-and-mine,-R- Ranch and Mine,NaN,|Tourism|Entertainment|Games|,Tourism,"60,000",operating,USA,TX,Dallas,Fort Worth,2.0,2014-01-01,2014-01,2014-Q1,2014.0,2014-08-17,2014-09-26,0.0,0.0,60000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,No Series A-H
5,/organization/club-domains,.Club Domains,http://nic.club/,|Software|,Software,"70,00,000",NaN,USA,FL,Ft. Lauderdale,Oakland Park,1.0,2011-10-10,2011-10,2011-Q4,2011.0,2013-05-31,2013-05-31,0.0,7000000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7000000.0,0.0,0.0,0.0,0.0,0.0,0.0,1,Series B
6,/organization/fox-networks,.Fox Networks,http://www.dotfox.com,|Advertising|,Advertising,"49,12,393",closed,ARG,NaN,Buenos Aires,Buenos Aires,1.0,NaN,NaN,NaN,NaN,2007-01-16,2007-01-16,0.0,0.0,0.0,4912393.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,No Series A-H
7,/organization/0-6-com,0-6.com,http://www.0-6.com,|Curated Web|,Curated Web,"20,00,000",operating,NaN,NaN,NaN,NaN,1.0,2007-01-01,2007-01,2007-Q1,2007.0,2008-03-19,2008-03-19,0.0,2000000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2000000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,Series A
8,/organization/004-technologies,004 Technologies,http://004gmbh.de/en/004-interact,|Software|,Software,-,operating,USA,IL,"Springfield, Illinois",Champaign,1.0,2010-01-01,2010-01,2010-Q1,2010.0,2014-07-24,2014-07-24,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,No Series A-H
9,/organization/01games-technology,01Games Technology,http://www.01games.hk/,|Games|,Games,"41,250",operating,HKG,NaN,Hong Kong,Hong Kong,1.0,NaN,NaN,NaN,NaN,2014-07-01,2014-07-01,41250.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,No Series A-H



TARGET DISTRIBUTION:


target
No Series A-H    36680
Series A          5722
Series B          3580
Series C          1925
Series D+         1531
Name: count, dtype: int64


UNIQUE VALUES IN CATEGORICAL COLUMNS:

permalink: 49436 unique values
permalink
/organization/prysm                               2
/organization/treasure-valley-urology-services    2
/organization/waywire                             1
/organization/tv-communications                   1
/organization/rock-your-paper                     1
/organization/in-touch-network                    1
/organization/r-ranch-and-mine                    1
/organization/club-domains                        1
/organization/fox-networks                        1
/organization/0-6-com                             1
Name: count, dtype: int64

name: 49350 unique values
name
Roost                 4
Spire                 4
Compass               3
Cue                   3
Hubbub                3
Peach                 3
Roadmap               3
Shift                 3
Adometry By Google    2
APS                   2
Name: count, dtype: int64

homepage_url: 45850 unique values
homepage_url
http://www.360fly.com      

In [18]:
# Check numerical columns
print("NUMERICAL COLUMN SUMMARY:")
display(funding.select_dtypes(include=["number"]).describe().T)

# Check date ranges
date_cols = [
    "founded_at",
    "first_funding_at",
    "last_funding_at"
]

print("\nDATE RANGES:")
for col in date_cols:
    dates = pd.to_datetime(funding[col], errors="coerce")
    print(f"\n{col}")
    print("Minimum:", dates.min())
    print("Maximum:", dates.max())
    print("Missing:", dates.isna().sum())

# Check funding round count by target
print("\nFUNDING ROUNDS BY TARGET:")
display(
    funding.groupby("target")["funding_rounds"]
    .agg(["count", "mean", "median", "min", "max"])
)

# Check target versus status
print("\nTARGET VS STATUS:")
display(pd.crosstab(funding["target"], funding["status"], margins=True))

NUMERICAL COLUMN SUMMARY:


,count,mean,std,min,25%,50%,75%,max
funding_rounds,49438.0,1.696205e+00,1.294213e+00,1.0,1.0,1.0,2.0,1.800000e+01
founded_year,38482.0,2.007359e+03,7.579203e+00,1902.0,2006.0,2010.0,2012.0,2.014000e+03
seed,49438.0,2.173215e+05,1.056985e+06,0.0,0.0,0.0,25000.0,1.300000e+08
venture,49438.0,7.501051e+06,2.847112e+07,0.0,0.0,0.0,5000000.0,2.351000e+09
equity_crowdfunding,49438.0,6.163322e+03,1.999048e+05,0.0,0.0,0.0,0.0,2.500000e+07
undisclosed,49438.0,1.302213e+05,2.981404e+06,0.0,0.0,0.0,0.0,2.924328e+08
convertible_note,49438.0,2.336410e+04,1.432046e+06,0.0,0.0,0.0,0.0,3.000000e+08
debt_financing,49438.0,1.888157e+06,1.382046e+08,0.0,0.0,0.0,0.0,3.007950e+10
angel,49438.0,6.541898e+04,6.582908e+05,0.0,0.0,0.0,0.0,6.359026e+07
grant,49438.0,1.628453e+05,5.612088e+06,0.0,0.0,0.0,0.0,7.505000e+08



DATE RANGES:

founded_at
Minimum: 1636-09-08 00:00:00
Maximum: 2014-12-13 00:00:00
Missing: 10884

first_funding_at
Minimum: 0001-05-14 00:00:00
Maximum: 2014-12-31 00:00:00
Missing: 0

last_funding_at
Minimum: 0001-05-14 00:00:00
Maximum: 2015-01-01 00:00:00
Missing: 0

FUNDING ROUNDS BY TARGET:


,count,mean,median,min,max
target,,,,,
No Series A-H,36680,1.390213,1.0,1.0,16.0
Series A,5722,1.864733,1.0,1.0,13.0
Series B,3580,2.577933,2.0,1.0,15.0
Series C,1925,3.256623,3.0,1.0,18.0
Series D+,1531,4.373612,4.0,1.0,15.0



TARGET VS STATUS:


status,acquired,closed,operating,All
target,,,,
No Series A-H,1755,1922,31954,35631
Series A,606,357,4604,5567
Series B,591,196,2721,3508
Series C,409,78,1413,1900
Series D+,331,50,1137,1518
All,3692,2603,41829,48124


In [19]:

# ============================================
# STARTUP FOLLOW-ON FUNDING CLASSIFIER
# PREPROCESSING
# ============================================

import os
import re
import numpy as np
import pandas as pd
import scipy.sparse as sp
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

# --------------------------------------------
# 1. SETTINGS
# --------------------------------------------

RANDOM_STATE = 42

OUTPUT_DIR = (
    r"C:\Users\Dell\Desktop\founders-carnet"
    r"\classifier_data\preprocessed"
    r"\startup_investment_classifier"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Check required DataFrame
if "funding" not in globals():
    raise NameError("DataFrame 'funding' was not found.")

required_columns = [
    "category_list", "market", "country_code",
    "state_code", "region", "city",
    "founded_year", "first_funding_at",
    "funding_rounds"
]

missing_columns = [
    col for col in required_columns
    if col not in funding.columns
]

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

# --------------------------------------------
# 2. COPY DATA AND CLEAN TARGET
# --------------------------------------------

df = funding.copy().reset_index(drop=True)

df["funding_rounds"] = pd.to_numeric(
    df["funding_rounds"], errors="coerce"
)

# Remove rows where target cannot be determined
df = df.dropna(subset=["funding_rounds"]).copy()

# 1 = More than one funding round
# 0 = One or zero funding rounds
df["target"] = (
    df["funding_rounds"] > 1
).astype(int)

# --------------------------------------------
# 3. CLEAN INVALID DATES
# --------------------------------------------

# Convert first funding date to datetime.
# Invalid dates, including year 0001, become NaT.
df["first_funding_at"] = pd.to_datetime(
    df["first_funding_at"],
    errors="coerce"
)

# Extract year and month
df["first_funding_year"] = (
    df["first_funding_at"].dt.year
)

df["first_funding_month"] = (
    df["first_funding_at"].dt.month
)

# Clean founded_year
df["founded_year"] = pd.to_numeric(
    df["founded_year"], errors="coerce"
)

# Remove implausible founding years
df.loc[
    (df["founded_year"] < 1800) |
    (df["founded_year"] > 2015),
    "founded_year"
] = np.nan

# --------------------------------------------
# 4. SELECT FEATURES
# --------------------------------------------

categorical_features = [
    "market",
    "country_code",
    "state_code",
    "region",
    "city"
]

numeric_features = [
    "founded_year",
    "first_funding_year",
    "first_funding_month"
]

text_feature = "category_list"

feature_columns = (
    [text_feature] +
    categorical_features +
    numeric_features
)

X = df[feature_columns].copy()
y = df["target"].copy()

# Clean text and categorical columns
for col in [text_feature] + categorical_features:
    X[col] = (
        X[col]
        .fillna("")
        .astype(str)
        .str.strip()
    )

# --------------------------------------------
# 5. SPLIT BEFORE FITTING TRANSFORMERS
# --------------------------------------------

# 70% train, 15% validation, 15% test

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

# --------------------------------------------
# 6. BUILD PREPROCESSING PIPELINE
# --------------------------------------------

text_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=10000,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
        token_pattern=r"(?u)\b\w+\b"
    ))
])

categorical_pipeline = Pipeline([
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=10
    ))
])

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("category_text", text_pipeline, text_feature),
        ("categorical", categorical_pipeline, categorical_features),
        ("numeric", numeric_pipeline, numeric_features)
    ],
    remainder="drop",
    sparse_threshold=1.0
)

# --------------------------------------------
# 7. FIT ON TRAINING DATA ONLY
# --------------------------------------------

X_train_processed = preprocessor.fit_transform(X_train)

X_val_processed = preprocessor.transform(X_val)

X_test_processed = preprocessor.transform(X_test)

# Ensure sparse matrices
X_train_processed = sp.csr_matrix(X_train_processed)
X_val_processed = sp.csr_matrix(X_val_processed)
X_test_processed = sp.csr_matrix(X_test_processed)

# --------------------------------------------
# 8. SAVE PROCESSED DATA
# --------------------------------------------

sp.save_npz(
    os.path.join(OUTPUT_DIR, "X_train.npz"),
    X_train_processed
)

sp.save_npz(
    os.path.join(OUTPUT_DIR, "X_val.npz"),
    X_val_processed
)

sp.save_npz(
    os.path.join(OUTPUT_DIR, "X_test.npz"),
    X_test_processed
)

# Save target labels
y_train.to_csv(
    os.path.join(OUTPUT_DIR, "y_train.csv"),
    index=False
)

y_val.to_csv(
    os.path.join(OUTPUT_DIR, "y_val.csv"),
    index=False
)

y_test.to_csv(
    os.path.join(OUTPUT_DIR, "y_test.csv"),
    index=False
)

# Save fitted preprocessing pipeline
joblib.dump(
    preprocessor,
    os.path.join(OUTPUT_DIR, "preprocessor.pkl")
)

# Save feature and split information
joblib.dump(
    {
        "feature_columns": feature_columns,
        "text_feature": text_feature,
        "categorical_features": categorical_features,
        "numeric_features": numeric_features,
        "target_name": "target",
        "target_mapping": {
            0: "No follow-on funding",
            1: "Follow-on funding"
        },
        "random_state": RANDOM_STATE
    },
    os.path.join(OUTPUT_DIR, "metadata.pkl")
)

# Save cleaned dataset
df.to_csv(
    os.path.join(OUTPUT_DIR, "cleaned_funding.csv"),
    index=False
)

# --------------------------------------------
# 9. DISPLAY RESULTS
# --------------------------------------------

print("=" * 55)
print("STARTUP FOLLOW-ON FUNDING PREPROCESSING COMPLETE")
print("=" * 55)

print("\nOriginal dataset shape:", funding.shape)
print("Final dataset shape:", df.shape)

print("\nTarget distribution:")
print(
    df["target"]
    .value_counts()
    .sort_index()
    .rename(index={
        0: "No follow-on funding",
        1: "Follow-on funding"
    })
)

print("\nTarget percentages:")
print(
    (df["target"].value_counts(normalize=True) * 100)
    .sort_index()
    .round(2)
)

print("\nSplit sizes:")
print("Training:", X_train.shape[0])
print("Validation:", X_val.shape[0])
print("Testing:", X_test.shape[0])

print("\nProcessed matrix shapes:")
print("Training:", X_train_processed.shape)
print("Validation:", X_val_processed.shape)
print("Testing:", X_test_processed.shape)

print("\nInvalid first funding dates:", 
      df["first_funding_at"].isna().sum())

print("\nSaved files:")
for file in sorted(os.listdir(OUTPUT_DIR)):
    print("-", file)

print("\nOutput directory:", OUTPUT_DIR)
print("=" * 55)


STARTUP FOLLOW-ON FUNDING PREPROCESSING COMPLETE

Original dataset shape: (49438, 41)
Final dataset shape: (49438, 43)

Target distribution:
target
No follow-on funding    32039
Follow-on funding       17399
Name: count, dtype: int64

Target percentages:
target
0    64.81
1    35.19
Name: proportion, dtype: float64

Split sizes:
Training: 34606
Validation: 7416
Testing: 7416

Processed matrix shapes:
Training: (34606, 7597)
Validation: (7416, 7597)
Testing: (7416, 7597)

Invalid first funding dates: 0

Saved files:
- X_test.npz
- X_train.npz
- X_val.npz
- cleaned_funding.csv
- metadata.pkl
- preprocessor.pkl
- y_test.csv
- y_train.csv
- y_val.csv

Output directory: C:\Users\Dell\Desktop\founders-carnet\classifier_data\preprocessed\startup_investment_classifier


In [20]:
import os
import scipy.sparse as sp

OUTPUT_DIR = (
    r"C:\Users\Dell\Desktop\founders-carnet"
    r"\classifier_data\preprocessed"
    r"\startup_investment_classifier"
)

for name in ["X_train.npz", "X_val.npz", "X_test.npz"]:
    matrix = sp.load_npz(os.path.join(OUTPUT_DIR, name))
    print(name, matrix.shape)

print("\nSaved files:")
for file in sorted(os.listdir(OUTPUT_DIR)):
    print(file)

X_train.npz (34606, 7597)
X_val.npz (7416, 7597)
X_test.npz (7416, 7597)

Saved files:
X_test.npz
X_train.npz
X_val.npz
cleaned_funding.csv
metadata.pkl
preprocessor.pkl
y_test.csv
y_train.csv
y_val.csv
